## Import Library


In [ ]:
!pip install -q gdown
import gdown

In [48]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity


## Overview

In [49]:
file_id = "1ABCxyz123456"
gdown.download(
    f"https://drive.google.com/uc?id={file_id}",
    "spotify_data.csv",
    quiet=False
)

df = pd.read_csv('spotify_data.csv')
df.head(10)

,Unnamed: 0,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
0,0,Jason Mraz,I Won't Give Up,53QF56cjZA9RTuuMZDrSA6,68,2012,acoustic,0.483,0.303,4,-10.058,1,0.0429,0.6940,0.000000,0.1150,0.139,133.406,240166,3
1,1,Jason Mraz,93 Million Miles,1s8tP3jP4GZcyHDsjvw218,50,2012,acoustic,0.572,0.454,3,-10.286,1,0.0258,0.4770,0.000014,0.0974,0.515,140.182,216387,4
2,2,Joshua Hyslop,Do Not Let Me Go,7BRCa8MPiyuvr2VU3O9W0F,57,2012,acoustic,0.409,0.234,3,-13.711,1,0.0323,0.3380,0.000050,0.0895,0.145,139.832,158960,4
3,3,Boyce Avenue,Fast Car,63wsZUhUZLlh1OsyrZq7sz,58,2012,acoustic,0.392,0.251,10,-9.845,1,0.0363,0.8070,0.000000,0.0797,0.508,204.961,304293,4
4,4,Andrew Belle,Sky's Still Blue,6nXIYClvJAfi6ujLiKqEq8,54,2012,acoustic,0.430,0.791,6,-5.419,0,0.0302,0.0726,0.019300,0.1100,0.217,171.864,244320,4
5,5,Chris Smither,What They Say,24NvptbNKGs6sPy1Vh1O0v,48,2012,acoustic,0.566,0.570,2,-6.420,1,0.0329,0.6880,0.000002,0.0943,0.960,83.403,166240,4
6,6,Matt Wertz,Walking in a Winter Wonderland,0BP7hSvLAG3URGrEvNNbGM,48,2012,acoustic,0.575,0.606,9,-8.197,1,0.0300,0.0119,0.000000,0.0675,0.364,121.083,152307,4
7,7,Green River Ordinance,Dancing Shoes,3Y6BuzQCg9p4yH347Nn8OW,45,2012,acoustic,0.586,0.423,7,-7.459,1,0.0261,0.2520,0.000006,0.0976,0.318,138.133,232373,4
8,8,Jason Mraz,Living in the Moment,3ce7k1L4EkZppZPz1EJWTS,44,2012,acoustic,0.650,0.628,7,-7.160,1,0.0232,0.0483,0.000000,0.1190,0.700,84.141,235080,4
9,9,Boyce Avenue,Heaven,2EKxmYmUdAVXlaHCnnW13o,58,2012,acoustic,0.619,0.280,8,-10.238,0,0.0317,0.7300,0.000000,0.1030,0.292,129.948,250063,4


## Data Cleaning

In [50]:
df = df.drop(columns=['Unnamed: 0','track_id','artist_name','year'])

### Null

In [51]:
df.isna().sum()

track_name          1
popularity          0
genre               0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
duration_ms         0
time_signature      0
dtype: int64

In [52]:
df = df.dropna()

### Duplicated

In [53]:
df.duplicated().sum()

np.int64(218)

### Out of range

In [54]:
df.describe()

,popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
count,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06,1.159763e+06
mean,1.838313e+01,5.374384e-01,6.396696e-01,5.287777e+00,-8.981356e+00,6.346529e-01,9.281477e-02,3.215372e-01,2.523491e-01,2.230190e-01,4.555639e-01,1.213772e+02,2.495618e+05,3.885879e+00
std,1.588555e+01,1.844780e-01,2.705009e-01,3.555198e+00,5.682216e+00,4.815276e-01,1.268410e-01,3.549872e-01,3.650732e-01,2.010708e-01,2.685189e-01,2.977975e+01,1.494262e+05,4.676969e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-5.810000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.073000e+03,0.000000e+00
25%,5.000000e+00,4.130000e-01,4.540000e-01,2.000000e+00,-1.082900e+01,0.000000e+00,3.710000e-02,6.400000e-03,1.050000e-06,9.790000e-02,2.260000e-01,9.879700e+01,1.810910e+05,4.000000e+00
50%,1.500000e+01,5.500000e-01,6.940000e-01,5.000000e+00,-7.450000e+00,1.000000e+00,5.070000e-02,1.470000e-01,1.760000e-03,1.340000e-01,4.380000e-01,1.219310e+02,2.257440e+05,4.000000e+00
75%,2.900000e+01,6.770000e-01,8.730000e-01,8.000000e+00,-5.276000e+00,1.000000e+00,8.900000e-02,6.400000e-01,6.140000e-01,2.920000e-01,6.740000e-01,1.399030e+02,2.869140e+05,4.000000e+00
max,1.000000e+02,9.930000e-01,1.000000e+00,1.100000e+01,6.172000e+00,1.000000e+00,9.710000e-01,9.960000e-01,1.000000e+00,1.000000e+00,1.000000e+00,2.499930e+02,6.000495e+06,5.000000e+00


In [55]:
df.min()

track_name                 !
popularity                 0
genre               acoustic
danceability             0.0
energy                   0.0
key                        0
loudness               -58.1
mode                       0
speechiness              0.0
acousticness             0.0
instrumentalness         0.0
liveness                 0.0
valence                  0.0
tempo                    0.0
duration_ms             2073
time_signature             0
dtype: object

In [56]:
df.max()

track_name          ﻿sonate Nr. 16 in C-dur Kv 545: Rondo
popularity                                            100
genre                                            trip-hop
danceability                                        0.993
energy                                                1.0
key                                                    11
loudness                                            6.172
mode                                                    1
speechiness                                         0.971
acousticness                                        0.996
instrumentalness                                      1.0
liveness                                              1.0
valence                                               1.0
tempo                                             249.993
duration_ms                                       6000495
time_signature                                          5
dtype: object

# Standard Scaler

In [57]:
scaler = StandardScaler()
num_cols = ['popularity','danceability','energy','key','loudness','mode','speechiness','acousticness','instrumentalness','liveness','valence','tempo','duration_ms','time_signature']
df[num_cols] = scaler.fit_transform(df[num_cols])

In [58]:
encoder = OneHotEncoder()
genre_encode = encoder.fit_transform(df[['genre']])

In [59]:
genre_df = pd.DataFrame(genre_encode.toarray(), columns=encoder.get_feature_names_out(), index=df.index)
df = pd.concat([df, genre_df], axis=1)
df = df.drop(columns=['genre'])

In [60]:
df

,track_name,popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,...,genre_ska,genre_sleep,genre_songwriter,genre_soul,genre_spanish,genre_swedish,genre_tango,genre_techno,genre_trance,genre_trip-hop
0,I Won't Give Up,3.123398,-0.295094,-1.244616,-0.362224,-0.189476,0.758725,-0.393523,1.049229,-0.691229,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,93 Million Miles,1.990292,0.187348,-0.686392,-0.643502,-0.229601,0.758725,-0.528337,0.437939,-0.691192,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Do Not Let Me Go,2.430944,-0.696226,-1.499699,-0.643502,-0.832359,0.758725,-0.477092,0.046376,-0.691092,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Fast Car,2.493895,-0.788378,-1.436852,1.325447,-0.151991,0.758725,-0.445556,1.367550,-0.691229,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Sky's Still Blue,2.242093,-0.582391,0.559445,0.200333,0.626931,-1.318000,-0.493648,-0.701257,-0.638363,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1159759,Black Spirits,-0.905423,-0.891371,0.378300,1.325447,0.444960,-1.318000,-0.151487,0.009755,-0.690843,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1159760,Quiet Dawn,-0.968373,-0.116211,0.130611,0.481611,0.245214,-1.318000,-0.474727,1.314027,-0.690876,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1159761,Morning Ms Candis,-1.031323,-0.251729,-0.738148,-0.080945,0.082601,0.758725,-0.515723,0.437939,-0.682655,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1159762,Happy Christmas (War Is Over),-1.157224,-0.311356,-0.867538,-1.487337,-0.767596,0.758725,-0.514146,0.308357,-0.691056,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


# Cosin Similarity

In [69]:
name = 'Dancing Shoes'
song = df[df['track_name'] == name]
song_col = song.drop(columns=['track_name'])
new_df = df.drop(columns=['track_name'])
similarity = cosine_similarity(song_col, new_df)

In [70]:
sml = similarity[0].argsort()[::-1]

df.iloc[sml[0:11]][["track_name"]]

,track_name
7,Dancing Shoes
926189,Be Mine - Remix
54820,Chill In The Air
530485,Good Enough
476966,Ba't Ganto Ang Pag-ibig
476780,What I'm Here For
309322,Old Friends
253168,See You Again
1066476,Static Waves
1066494,God Willin' & the Creek Don't Rise (with The P...
